# Notebook 2: Federated Learning Experiment Sweep — IDRiD Dataset

**Metrics implemented:**
- `global_accuracy`: Global model evaluated on shared 103-image test set (primary FL metric)
- `client_X_local_on_test_accuracy`: **PRIMARY fairness metric** — each client's LOCAL model (just trained, before aggregation) evaluated on the shared test set. Consistent with RetinaMNIST experiments.
- `client_X_drift_diagnostic`: Secondary — global model evaluated on client's training data. Measures how well the global model covers each client's local distribution.

In [1]:
import sys, os, time, torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Python: {sys.version}')
print(f'PyTorch: {torch.__version__}')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/(1024**3):.2f} GB')

Python: 3.10.12 (main, Jun 11 2023, 05:26:28) [GCC 11.4.0]
PyTorch: 2.5.1+cu121
Device: cuda
GPU: NVIDIA RTX A2000 12GB
VRAM: 11.62 GB


In [2]:
import pandas as pd
NOTEBOOK_DIR = os.getcwd()
DATA_DIR = os.path.join(NOTEBOOK_DIR, 'data', 'idrid_raw', 'B. Disease Grading')
train_csv = os.path.join(DATA_DIR, '2. Groundtruths', 'a. IDRiD_Disease Grading_Training Labels.csv')
test_csv  = os.path.join(DATA_DIR, '2. Groundtruths', 'b. IDRiD_Disease Grading_Testing Labels.csv')
train_df = pd.read_csv(train_csv)[['Image name','Retinopathy grade']].dropna()
train_df['Retinopathy grade'] = train_df['Retinopathy grade'].astype(int)
test_df  = pd.read_csv(test_csv)[['Image name','Retinopathy grade']].dropna()
test_df['Retinopathy grade']  = test_df['Retinopathy grade'].astype(int)
image_names = train_df['Image name'].values
labels      = train_df['Retinopathy grade'].values
train_img_dir = os.path.join(DATA_DIR, '1. Original Images', 'a. Training Set')
test_img_dir  = os.path.join(DATA_DIR, '1. Original Images', 'b. Testing Set')
print(f'Train: {len(train_df)} | Test: {len(test_df)}')

Train: 413 | Test: 103


In [3]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms

class IDRiDDataset(Dataset):
    def __init__(self, image_names, labels, img_dir, transform=None):
        self.image_names = list(image_names)
        self.labels = list(labels)
        self.img_dir = img_dir
        self.transform = transform
    def __len__(self): return len(self.image_names)
    def __getitem__(self, idx):
        img = Image.open(os.path.join(self.img_dir, self.image_names[idx]+'.jpg')).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, int(self.labels[idx])

train_transform = transforms.Compose([
    transforms.Resize((240,240)),
    transforms.RandomCrop((224,224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
eval_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
print('Transforms ready.')

Transforms ready.


In [4]:
import random, numpy as np

def partition_dirichlet(image_names, labels, num_clients, alpha, seed):
    np.random.seed(seed); random.seed(seed); torch.manual_seed(seed)
    image_names = np.array(image_names); labels = np.array(labels)
    num_classes = len(np.unique(labels))
    client_indices = [[] for _ in range(num_clients)]
    for cls in range(num_classes):
        idxs = np.where(labels == cls)[0]; np.random.shuffle(idxs)
        props = np.random.dirichlet([alpha]*num_clients)
        splits = (props * len(idxs)).astype(int)
        splits[-1] = len(idxs) - splits[:-1].sum()
        start = 0
        for cid, s in enumerate(splits):
            client_indices[cid].extend(idxs[start:start+s].tolist()); start += s
    partitions = []
    for cid in range(num_clients):
        idxs = client_indices[cid]; np.random.shuffle(idxs)
        partitions.append((image_names[idxs].tolist(), labels[idxs].tolist()))
    return partitions

print('Partitioner ready.')

Partitioner ready.


In [5]:
import json, torch.nn as nn, timm

def get_weights(model):
    return [v.cpu().numpy() for v in model.state_dict().values()]

def set_weights(model, weights):
    sd = dict(zip(model.state_dict().keys(), [torch.tensor(w) for w in weights]))
    model.load_state_dict(sd, strict=True)

def train_fedprox(model, dataloader, global_weights, mu, lr, epochs, device):
    gsd = dict(zip(model.state_dict().keys(), global_weights))
    gparams = [torch.tensor(gsd[n]).to(device) for n, _ in model.named_parameters()]
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    crit = nn.CrossEntropyLoss()
    model.train()
    for _ in range(epochs):
        for bx, by in dataloader:
            bx, by = bx.to(device), by.to(device).long()
            opt.zero_grad()
            loss = crit(model(bx), by)
            if mu > 0:
                prox = sum(torch.norm(lp - gp)**2 for lp, gp in zip(model.parameters(), gparams))
                loss = loss + (mu/2)*prox
            loss.backward(); opt.step()
    return get_weights(model), len(dataloader.dataset)

def evaluate_model(model, dataloader, device):
    model.eval(); total_loss=0.0; correct=0; total=0
    crit = nn.CrossEntropyLoss()
    with torch.no_grad():
        for bx, by in dataloader:
            bx, by = bx.to(device), by.to(device).long()
            out = model(bx)
            total_loss += crit(out, by).item() * len(by)
            correct += (out.argmax(1)==by).sum().item(); total += len(by)
    return (correct/total, total_loss/total) if total>0 else (0.0, 0.0)

def compute_l2_drift(lw, gw):
    return float(np.sqrt(sum(np.sum((a-b)**2) for a,b in zip(lw,gw))))

COMPLETED_FILE = os.path.join(NOTEBOOK_DIR, 'checkpoints', 'completed.json')
def is_completed(eid):
    if not os.path.exists(COMPLETED_FILE): return False
    try:
        with open(COMPLETED_FILE) as f: return eid in json.load(f)
    except: return False

def mark_completed(eid):
    os.makedirs(os.path.dirname(COMPLETED_FILE), exist_ok=True)
    done = []
    if os.path.exists(COMPLETED_FILE):
        try:
            with open(COMPLETED_FILE) as f: done = json.load(f)
        except: pass
    done.append(eid)
    tmp = COMPLETED_FILE+'.tmp'
    with open(tmp,'w') as f: json.dump(done, f, indent=1)
    os.replace(tmp, COMPLETED_FILE)

print('FL utilities ready.')

FL utilities ready.


/home/shivansh/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
print('[INFO] Loading shared test dataset...')
test_dataset = IDRiDDataset(test_df['Image name'].values, test_df['Retinopathy grade'].values,
                            test_img_dir, transform=eval_transform)
test_loader  = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=4, pin_memory=True)
print(f'[OK] Test dataset loaded: {len(test_dataset)} images.')

[INFO] Loading shared test dataset...
[OK] Test dataset loaded: 103 images.


In [7]:
def run_experiment(alpha, mu, seed, strategy, config):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed(seed)

    partitions = partition_dirichlet(image_names, labels, config['num_clients'], alpha, seed)
    client_loaders = {}
    for cid in range(config['num_clients']):
        imgs, lbls = partitions[cid]
        if len(imgs) == 0:
            client_loaders[cid] = None
        else:
            ds = IDRiDDataset(imgs, lbls, train_img_dir, transform=train_transform)
            client_loaders[cid] = DataLoader(ds, batch_size=config['batch_size'],
                                             shuffle=True, num_workers=4, pin_memory=True)

    global_model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=5).to(device)
    global_weights = get_weights(global_model)
    round_metrics_list = []

    for rnd in range(1, config['num_rounds']+1):
        client_updates = []
        round_client_metrics = []

        for cid in range(config['num_clients']):
            loader = client_loaders[cid]
            if loader is None:
                client_updates.append((global_weights, 0))
                round_client_metrics.append({
                    'local_test_acc': 0.0, 'local_test_loss': 0.0,
                    'drift_diagnostic_acc': 0.0, 'drift_diagnostic_loss': 0.0,
                    'drift': 0.0, 'train_loss': 0.0
                })
                continue

            local_model = timm.create_model('efficientnet_b0', num_classes=5).to(device)
            set_weights(local_model, global_weights)

            local_weights, n_samples = train_fedprox(
                local_model, loader, global_weights,
                mu=mu, lr=config['lr'], epochs=config['local_epochs'], device=device
            )

            # PRIMARY FAIRNESS METRIC: local model (before aggregation) on shared test set
            local_test_acc, local_test_loss = evaluate_model(local_model, test_loader, device)

            # SECONDARY DRIFT DIAGNOSTIC: local training loss
            _, train_loss = evaluate_model(local_model, loader, device)
            drift = compute_l2_drift(local_weights, global_weights)

            client_updates.append((local_weights, n_samples))
            round_client_metrics.append({
                'local_test_acc': local_test_acc,
                'local_test_loss': local_test_loss,
                'drift_diagnostic_acc': 0.0,   # filled after aggregation below
                'drift_diagnostic_loss': 0.0,
                'drift': drift,
                'train_loss': train_loss
            })
            del local_model

        # Aggregate
        global_weights = (lambda updates: [
            sum(w[i]*(n/sum(nn for _,nn in updates)) for w,n in updates)
            for i in range(len(updates[0][0]))
        ])(client_updates)
        set_weights(global_model, global_weights)

        # Global model on shared test set
        global_acc, global_loss = evaluate_model(global_model, test_loader, device)

        # SECONDARY: global model on each client's training data (drift diagnostic)
        for cid in range(config['num_clients']):
            loader = client_loaders[cid]
            if loader is not None:
                d_acc, d_loss = evaluate_model(global_model, loader, device)
                round_client_metrics[cid]['drift_diagnostic_acc'] = d_acc
                round_client_metrics[cid]['drift_diagnostic_loss'] = d_loss

        mean_drift = float(np.mean([m['drift'] for m in round_client_metrics]))
        max_drift  = float(np.max( [m['drift'] for m in round_client_metrics]))
        mean_train = float(np.mean([m['train_loss'] for m in round_client_metrics]))

        round_metrics_list.append({
            'round': rnd,
            'global_accuracy': global_acc,
            'global_loss': global_loss,
            'mean_train_loss': mean_train,
            'mean_drift': mean_drift,
            'max_drift': max_drift,
            'client_metrics': round_client_metrics
        })

        mem = torch.cuda.memory_allocated(device)/(1024**2) if torch.cuda.is_available() else 0
        local_accs = [m['local_test_acc'] for m in round_client_metrics]
        print(f'  Round {rnd:2d}/{config["num_rounds"]} | '
              f'Global: {global_acc*100:.2f}% | '
              f'Client local-on-test: [{local_accs[0]*100:.1f}%, {local_accs[1]*100:.1f}%, {local_accs[2]*100:.1f}%] | '
              f'GPU: {mem:.0f}MB')

    final_local_accs = [m['local_test_acc'] for m in round_metrics_list[-1]['client_metrics']]
    final_acc  = round_metrics_list[-1]['global_accuracy']
    best_acc   = max(rm['global_accuracy'] for rm in round_metrics_list)
    conv = next((rm['round'] for rm in round_metrics_list if rm['global_accuracy'] >= 0.95*final_acc), None)

    del global_model
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    return {
        'alpha': alpha, 'mu': mu, 'seed': seed, 'strategy': strategy,
        'final_accuracy': final_acc, 'best_accuracy': best_acc,
        'convergence_round': conv,
        # PRIMARY fairness stats from local-on-test metric
        'accuracy_std': float(np.std(final_local_accs)),
        'worst_client_accuracy': float(np.min(final_local_accs)),
        'per_client_local_on_test': final_local_accs,
        'round_metrics': round_metrics_list
    }

print('Experiment runner ready.')

Experiment runner ready.


In [8]:
def save_experiment_results(result, base_dir='.'):
    rows = []
    for rm in result['round_metrics']:
        cm = rm['client_metrics']
        rows.append({
            'experiment_id': result['experiment_id'],
            'round': rm['round'],
            'global_accuracy': rm['global_accuracy'],
            'global_loss': rm['global_loss'],
            'mean_train_loss': rm['mean_train_loss'],
            'mean_drift': rm['mean_drift'],
            'max_drift': rm['max_drift'],
            # PRIMARY fairness metric
            'client_0_local_on_test_accuracy': cm[0]['local_test_acc'],
            'client_0_local_on_test_loss':     cm[0]['local_test_loss'],
            'client_1_local_on_test_accuracy': cm[1]['local_test_acc'],
            'client_1_local_on_test_loss':     cm[1]['local_test_loss'],
            'client_2_local_on_test_accuracy': cm[2]['local_test_acc'],
            'client_2_local_on_test_loss':     cm[2]['local_test_loss'],
            # SECONDARY drift diagnostic
            'client_0_drift_diagnostic': cm[0]['drift_diagnostic_acc'],
            'client_1_drift_diagnostic': cm[1]['drift_diagnostic_acc'],
            'client_2_drift_diagnostic': cm[2]['drift_diagnostic_acc'],
            'alpha': result['alpha'], 'mu': result['mu'],
            'seed': result['seed'], 'strategy': result['strategy'],
            'straggler_fraction': 0.0
        })
    rpath = os.path.join(base_dir, 'results', 'round_metrics.csv')
    os.makedirs(os.path.dirname(rpath), exist_ok=True)
    rdf = pd.DataFrame(rows)
    rdf.to_csv(rpath, mode='a', header=not os.path.exists(rpath), index=False)

    spath = os.path.join(base_dir, 'results', 'experiment_summary.csv')
    srow = {
        'experiment_id': result['experiment_id'],
        'alpha': result['alpha'], 'mu': result['mu'],
        'seed': result['seed'], 'strategy': result['strategy'],
        'straggler_fraction': 0.0,
        'final_global_accuracy': result['final_accuracy'],
        'best_global_accuracy': result['best_accuracy'],
        'convergence_round': result['convergence_round'],
        'local_on_test_accuracy_std': result['accuracy_std'],
        'worst_client_local_on_test_accuracy': result['worst_client_accuracy'],
        'per_client_local_on_test': str(result['per_client_local_on_test']),
        'runtime_seconds': result['runtime_seconds']
    }
    sdf = pd.DataFrame([srow])
    sdf.to_csv(spath, mode='a', header=not os.path.exists(spath), index=False)

print('Serialization ready.')

Serialization ready.


In [ ]:
import time

SWEEP_CONFIG = {
    'num_clients': 3, 'num_rounds': 10,
    'local_epochs': 3, 'batch_size': 8, 'lr': 1e-4
}

# Seed confirmation runs for alpha=0.3
all_configs = [
    {'alpha': 0.3, 'mu': 0.0, 'strategy': 'fedavg',  'seed': 123},
    {'alpha': 0.3, 'mu': 0.0, 'strategy': 'fedavg',  'seed': 777},
    {'alpha': 0.3, 'mu': 0.1, 'strategy': 'fedprox', 'seed': 123},
    {'alpha': 0.3, 'mu': 0.1, 'strategy': 'fedprox', 'seed': 777},
    {'alpha': 0.3, 'mu': 1.0, 'strategy': 'fedprox', 'seed': 123},
    {'alpha': 0.3, 'mu': 1.0, 'strategy': 'fedprox', 'seed': 777},
]

RUN_ONLY_FIRST = False

completed_count = 0; skipped_count = 0; runtimes = []
total_configs = len(all_configs)
print(f'[SWEEP] {total_configs} seed-confirmation configs for alpha=0.3')

for conf in all_configs:
    alpha, mu, seed, strategy = conf['alpha'], conf['mu'], conf['seed'], conf['strategy']
    exp_id = f'alpha{alpha}_mu{mu}_seed{seed}_{strategy}_straggler0.0'
    if is_completed(exp_id):
        print(f'[SKIP] {exp_id}'); skipped_count += 1; continue
    print(f'
{"="*60}')
    print(f'[RUN {completed_count+skipped_count+1}/{total_configs}] {exp_id}')
    print(f'{"="*60}')
    t0 = time.time()
    res = run_experiment(alpha=alpha, mu=mu, seed=seed, strategy=strategy, config=SWEEP_CONFIG)
    elapsed = time.time() - t0
    runtimes.append(elapsed)
    res['experiment_id'] = exp_id; res['runtime_seconds'] = elapsed
    save_experiment_results(res, NOTEBOOK_DIR)
    mark_completed(exp_id)
    completed_count += 1
    remaining = total_configs - (completed_count + skipped_count)
    eta = remaining * np.mean(runtimes)
    print(f'[DONE] {elapsed:.1f}s | Final global acc: {res["final_accuracy"]*100:.2f}%')
    print(f'[PROGRESS] {completed_count+skipped_count}/{total_configs} done | ETA: {eta/60:.1f} min')

print(f'
[SWEEP DONE] Ran: {completed_count} | Skipped: {skipped_count}')
